# Phase 6 — dual-attention MoE LORMA summarization


In [1]:
%pip install -q huggingface_hub tokenizers scikit-learn tqdm rouge-score


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from rouge_score import rouge_scorer
import time
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'dual_attention_200M'
MODEL_PATH = 'checkpoints/dual_attention_200M/model.pt'
TOKENIZER_PATH = 'tokenizer_dual_attention/tokenizer.json'
MODEL_CONFIG = {'d_model': 512, 'n_heads': 8, 'n_kv_heads': 4, 'n_layers': 12, 'd_ff': 2048, 'max_seq_len': 2048, 'n_experts': 8, 'top_k': 2, 'balance_alpha': 0.003}
MAX_SEQ_LEN = 2048
USE_GRADIENT_CHECKPOINTING = False
UNFREEZE_LAST_N_BLOCKS = 0  # pure LORMA, as in the supplied notebook
LORMA_LAST_N_BLOCKS = 2  # same attention targets as the supplied notebook
MAX_NEW_TOKENS = 512  # includes the EOS target; fixed source budget at train/test
SOURCE_SPANS = 8  # evenly spaced non-overlapping source passages
VAL_FRACTION = 0.05
VAL_MAX_DOCS = 100  # fixed held-out subset used for generation-based selection; None = full dev
LOG_EVERY_DOCS = 1000
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_6_Summ'
SUMMARY_KEY = 'summary'  # set explicitly if the dataset uses a different field
SKIP_EMPTY_SUMMARIES = True  # null/blank training targets cannot supervise generation
WARMUP_RATIO = 0.05
WARMUP_START_FACTOR = 0.1
LR_FACTOR = 0.5
LR_BAD_CHECKS = 2
LR_COOLDOWN_CHECKS = 1
LR_MIN = 1e-5
LR_LOSS_THRESHOLD = 0.001  # require 0.1% relative dev-loss improvement
VAL_LOSS_CHECKS_PER_EPOCH = 4
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG['max_seq_len']
assert 1 < MAX_NEW_TOKENS < MAX_SEQ_LEN
assert SOURCE_SPANS > 0 and LORMA_LAST_N_BLOCKS <= MODEL_CONFIG['n_layers']
assert UNFREEZE_LAST_N_BLOCKS == 0  # pure LoRMA
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5  # retained from the reference; lower this to cap training time
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(r=16, alpha=32, dropout=0.05, do_pi=True)
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'lorma'
TRAIN_FRACTION = 1.0  # use all training documents after holding out validation

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = None  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = False  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/summ_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'summ_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'summ_{PHASE1_TAG}_{TECHNIQUE}_len{MAX_SEQ_LEN}_summary{MAX_NEW_TOKENS}_spans{SOURCE_SPANS}_tokenroute_warmup_plateau'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)
if any(str(p).startswith('SET_') for p in (PHASE1_TAG, MODEL_PATH, TOKENIZER_PATH)):
    raise ValueError('Set the exact dual-attention Phase 1 tag, model checkpoint path and matching tokenizer path. The supplied reference notebook does not specify them.')

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = experiment_metadata('SUMM')
    manifest.update(save_tag=SAVE_TAG, best_tag=BEST_TAG, submission_zip=SUBMISSION_ZIP_NAME)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Total context / summary budget:', MAX_SEQ_LEN, MAX_NEW_TOKENS)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/dual_attention_200M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_dual_attention/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau_best/checkpoint.pt
Submission ZIP: /kaggle/working/summ_runs/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau.zip
Total context / summary budget: 2048 512


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


EOS_ID = EMPTY_ID
PROMPT_START = 'Document:\n'
PROMPT_END = '\n\nSummary:\n'
PREFIX_IDS = tokenizer.encode(PROMPT_START, add_special_tokens=False).ids
SUFFIX_IDS = tokenizer.encode(PROMPT_END, add_special_tokens=False).ids
PASSAGE_GAP_IDS = tokenizer.encode('\n[...]\n', add_special_tokens=False).ids
SOURCE_BUDGET = MAX_SEQ_LEN - MAX_NEW_TOKENS - len(PREFIX_IDS) - len(SUFFIX_IDS)
assert SOURCE_BUDGET >= SOURCE_SPANS
print('Document token budget:', SOURCE_BUDGET)


(…)/tokenizer_dual_attention/tokenizer.json:   0%|          | 0.00/2.54M [00:00<?, ?B/s]

Document token budget: 1524


In [4]:
def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'document' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> document object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected list of document objects')
    result, excluded = [], []
    seen_ids = set()
    for doc in docs:
        doc_id, text = doc['id'], doc['document']
        if not isinstance(doc_id, str) or not isinstance(text, str) or not text.strip():
            raise ValueError('Expected string ID and nonempty document text')
        if doc_id in seen_ids: raise ValueError(f'Duplicate document ID: {doc_id}')
        seen_ids.add(doc_id)
        row = dict(id=doc_id, document=text)
        if labelled:
            if SUMMARY_KEY not in doc:
                raise ValueError(f'{doc_id}: summary field {SUMMARY_KEY!r} is absent. Available fields: {list(doc)}. Set SUMMARY_KEY to the actual reference field; missing fields are not silently skipped.')
            target = doc[SUMMARY_KEY]
            if target is None or (isinstance(target, str) and not target.strip()):
                if not SKIP_EMPTY_SUMMARIES:
                    raise ValueError(f'{doc_id}: null or blank reference summary')
                excluded.append(dict(id=doc_id, reason='null' if target is None else 'blank'))
                continue
            if not isinstance(target, str):
                raise ValueError(f'{doc_id}: {SUMMARY_KEY!r} must contain a string; got {type(target).__name__}. Inspect this record rather than coercing the target.')
            row['summary'] = target
        result.append(row)
    if labelled:
        print(f'Training input records: {len(docs):,} | usable references: {len(result):,} | excluded null/blank summaries: {len(excluded):,}')
        if excluded: print('Excluded IDs (first 20):', [r['id'] for r in excluded[:20]])
        with open(WORK_DIR / 'excluded_training_summaries.json', 'w') as f:
            json.dump(excluded, f, indent=2)
    if not result: raise ValueError('No usable records remain; inspect the dataset reference field')
    return result

all_train = normalize_documents(download_json(f'{DATA_PREFIX}/Train/train.json'), labelled=True)
# Exact duplicate document texts stay together, even if IDs differ.
groups = {}
for row in all_train:
    digest = hashlib.sha256(' '.join(row['document'].split()).encode()).hexdigest()
    groups.setdefault(digest, []).append(row)
keys = sorted(groups)
random.Random(SEED).shuffle(keys)
if len(keys) < 2: raise ValueError('At least two distinct documents are required')
assert 0 < VAL_FRACTION < 1 and 0 < TRAIN_FRACTION <= 1
n_dev = min(len(keys)-1, max(1, round(len(keys)*VAL_FRACTION)))
dev_keys, train_keys = keys[:n_dev], keys[n_dev:]
train_keys = train_keys[:max(1, round(len(train_keys)*TRAIN_FRACTION))]
data = {'train': [d for k in train_keys for d in groups[k]],
        'dev': [d for k in dev_keys for d in groups[k]]}
# Stable fixed subset, independent of generated outputs.
if VAL_MAX_DOCS is not None: data['dev'] = data['dev'][:VAL_MAX_DOCS]
assert data['dev'] and not ({d['id'] for d in data['train']} & {d['id'] for d in data['dev']})
DATA_FINGERPRINT = hashlib.sha256(json.dumps(all_train, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
with open(WORK_DIR / 'split_ids.json', 'w') as f:
    json.dump({s: [d['id'] for d in rows] for s, rows in data.items()}, f, indent=2)
print('Public training records:', len(all_train), '| train:', len(data['train']), '| validation used:', len(data['dev']))
del all_train, groups


Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Training input records: 7,030 | usable references: 7,028 | excluded null/blank summaries: 2
Excluded IDs (first 20): ['299', '4799']
Public training records: 7028 | train: 6676 | validation used: 100


In [5]:
def select_source_tokens(tokens, budget=SOURCE_BUDGET, spans=SOURCE_SPANS):
    if len(tokens) <= budget: return list(tokens)
    spans = min(spans, budget)
    content_budget = budget - (spans - 1) * len(PASSAGE_GAP_IDS)
    if content_budget < spans: raise ValueError('Source budget too small for passage markers')
    widths = [content_budget // spans + int(i < content_budget % spans) for i in range(spans)]
    # Distribute omitted tokens between passages; retain document start and end.
    omitted = len(tokens) - content_budget
    if spans == 1: return tokens[:budget]
    selected, consumed = [], 0
    for i, width in enumerate(widths):
        start = consumed + (i * omitted // (spans - 1))
        if i: selected.extend(PASSAGE_GAP_IDS)
        selected.extend(tokens[start:start+width])
        consumed += width
    assert len(selected) == budget
    return selected

class SummaryDataset(Dataset):
    def __init__(self, documents, split):
        self.rows, self.split = [], split
        source_lengths, target_lengths = [], []
        for doc in tqdm(documents, desc=f'Tokenizing {split}'):
            source = tokenizer.encode(doc['document'], add_special_tokens=False).ids
            prompt = PREFIX_IDS + select_source_tokens(source) + SUFFIX_IDS
            row = dict(doc_id=doc['id'], prompt=prompt)
            source_lengths.append(len(source))
            if 'summary' in doc:
                target = tokenizer.encode(doc['summary'], add_special_tokens=False).ids
                if not target: raise ValueError(f"Empty tokenized summary: {doc['id']}")
                target_lengths.append(len(target) + 1)
                row.update(target=target[:MAX_NEW_TOKENS-1] + [EOS_ID], reference=doc['summary'])
            self.rows.append(row)
        for name, lengths in [('document', source_lengths), ('summary including EOS', target_lengths)]:
            if lengths:
                print(split, name, 'tokens p50/p90/p99/max:', torch.quantile(torch.tensor(lengths).float(), torch.tensor([.5,.9,.99,1.])).tolist())
        print(split, 'documents needing source selection:', sum(n > SOURCE_BUDGET for n in source_lengths))
        if target_lengths: print(split, 'reference summaries truncated for training/loss:', sum(n > MAX_NEW_TOKENS for n in target_lengths), '/', len(target_lengths))
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate_summary(rows):
    sequences = [r['prompt'] + r['target'] for r in rows]
    width = max(map(len, sequences))
    assert width <= MAX_SEQ_LEN
    ids = torch.full((len(rows), width), PAD_ID, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    for i, (r, sequence) in enumerate(zip(rows, sequences)):
        ids[i, :len(sequence)] = torch.tensor(sequence)
        labels[i, len(r['prompt']):len(sequence)] = torch.tensor(r['target'])
    return dict(input_ids=ids, labels=labels, valid_mask=torch.arange(width)[None, :] < torch.tensor([len(s) for s in sequences])[:, None], doc_ids=[r['doc_id'] for r in rows])

datasets = {s: SummaryDataset(rows, s) for s, rows in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_summary, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing train:   0%|          | 0/6676 [00:00<?, ?it/s]

train document tokens p50/p90/p99/max: [3635.0, 9050.5, 27528.75, 160451.0]
train summary including EOS tokens p50/p90/p99/max: [748.0, 1769.0, 5044.75, 32288.0]
train documents needing source selection: 6058
train reference summaries truncated for training/loss: 4876 / 6676


Tokenizing dev:   0%|          | 0/100 [00:00<?, ?it/s]

dev document tokens p50/p90/p99/max: [3763.5, 10661.498046875, 36186.81640625, 103969.0]
dev summary including EOS tokens p50/p90/p99/max: [743.0, 1593.8997802734375, 7982.81640625, 11033.0]
dev documents needing source selection: 93
dev reference summaries truncated for training/loss: 80 / 100


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (
                torch.arange(0, head_dim, 2).float() / head_dim
            )
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer(
            "cos",
            freqs.cos(),
            persistent=False,
        )

        self.register_buffer(
            "sin",
            freqs.sin(),
            persistent=False,
        )

    def forward(self, q, k):

        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [
                q1 * cos - q2 * sin,
                q1 * sin + q2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [
                k1 * cos - k2 * sin,
                k1 * sin + k2 * cos,
            ],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU Expert
# ============================================================

class SwiGLU(nn.Module):

    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.up_proj = nn.Linear(
            d_model,
            d_ff,
            bias=False,
        )

        self.down_proj = nn.Linear(
            d_ff,
            d_model,
            bias=False,
        )

    def forward(self, x):

        return self.down_proj(
            F.silu(self.gate_proj(x))
            * self.up_proj(x)
        )


# ============================================================
# Mixture of Experts
# ============================================================
class MoE(nn.Module):

    def __init__(
        self,
        d_model,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        # Router chooses experts BEFORE any expert is evaluated
        self.router = nn.Linear(
            d_model,
            n_experts,
            bias=False,
        )

        expert_d_ff = d_ff // top_k

        self.experts = nn.ModuleList([
            SwiGLU(
                d_model,
                expert_d_ff,
            )
            for _ in range(n_experts)
        ])

    def forward(self, x, valid_mask=None):

        # x: [B, T, D]

        # ------------------------------------------------
        # Batch representation
        # ------------------------------------------------

        batch_repr = (x.mean(dim=(0, 1)) if valid_mask is None else
            (x * valid_mask[..., None]).sum(dim=(0, 1)) / valid_mask.sum().clamp_min(1))  # [D]

        # ------------------------------------------------
        # ROUTE FIRST
        # ------------------------------------------------

        router_logits = self.router(
            batch_repr
        )  # [E]

        router_probs = F.softmax(
            router_logits,
            dim=-1,
        )

        topk_probs, topk_indices = torch.topk(
            router_probs,
            self.top_k,
            dim=-1,
        )

        # Normalize selected weights
        topk_probs = topk_probs / (
            topk_probs.sum() + 1e-9
        )

        # ------------------------------------------------
        # ONLY NOW evaluate selected experts
        # ------------------------------------------------

        output = 0.0

        for i in range(self.top_k):

            expert_idx = topk_indices[i].item()
            weight = topk_probs[i]

            expert_output = self.experts[
                expert_idx
            ](x)

            output = (
                output
                + weight * expert_output
            )

        # ------------------------------------------------
        # Batch routing loss
        # ------------------------------------------------

        # Router probability distribution for this batch.
        # Used to discourage permanent collapse to
        # a small subset of experts.
        balance_loss = (
            self.balance_alpha
            * self.n_experts
            * torch.sum(
                router_probs
                * router_probs
            )
        )

        return output, balance_loss

# ============================================================
# GQA + SDPA
# ============================================================
class GQAAttention(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        local_window=64,
        global_stride=128,
        max_seq_len=2048,
    ):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.local_window = local_window
        self.global_stride = global_stride
        self.max_seq_len = max_seq_len

        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def _build_sparse_mask(self, T, device):
        """
        Allowed attention for query position i:

        1. Previous local_window positions
        2. All previous global positions
           (every global_stride tokens)
        3. Itself

        Future positions are never allowed.
        """

        positions = torch.arange(
            T,
            device=device,
        )

        # i = query position
        # j = key position
        i = positions[:, None]
        j = positions[None, :]

        # Causal condition
        causal = j <= i

        # Local attention
        local = j >= (i - self.local_window + 1)

        # Global landmark positions
        global_pos = (j % self.global_stride) == 0

        # Allowed if local OR global, while remaining causal
        allowed = causal & (local | global_pos)

        # SDPA bool mask:
        # True  = allowed to attend
        # False = masked
        return allowed

    def forward(self, x, valid_mask=None):

        B, T, _ = x.shape

        # --------------------------------------------------
        # Q, K, V projections
        # --------------------------------------------------

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # --------------------------------------------------
        # Reshape into heads
        # --------------------------------------------------

        q = q.view(
            B,
            T,
            self.n_heads,
            self.head_dim,
        ).transpose(1, 2)

        k = k.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        v = v.view(
            B,
            T,
            self.n_kv_heads,
            self.head_dim,
        ).transpose(1, 2)

        # --------------------------------------------------
        # RoPE
        # --------------------------------------------------

        q, k = self.rope(q, k)

        # --------------------------------------------------
        # GQA: expand K/V heads
        # --------------------------------------------------

        repeat_factor = (
            self.n_heads // self.n_kv_heads
        )

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # --------------------------------------------------
        # Local + Global sparse attention
        # --------------------------------------------------

        attn_mask = self._build_sparse_mask(
            T,
            x.device,
        )

        if valid_mask is not None:
            attn_mask = attn_mask[None, None] & valid_mask[:, None, None, :]

        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            is_causal=False,
        )

        # --------------------------------------------------
        # Merge heads
        # --------------------------------------------------

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block with MoE
# ============================================================

class DecoderBlock(nn.Module):

    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(
            d_model
        )

        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
        )

        self.norm2 = nn.RMSNorm(
            d_model
        )

        self.moe = CausalMoE(
            d_model=d_model,
            d_ff=d_ff,
            n_experts=n_experts,
            top_k=top_k,
            balance_alpha=balance_alpha,
        )

    def forward(self, x, valid_mask=None):

        # Attention
        x = x + self.attention(
            self.norm1(x), valid_mask=valid_mask
        )

        # MoE FFN
        moe_output, balance_loss = self.moe(
            self.norm2(x), valid_mask=valid_mask
        )

        x = x + moe_output

        return x, balance_loss


# ============================================================
# Decoder-only MoE Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,

        # MoE
        n_experts=8,
        top_k=2,
        balance_alpha=0.003,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.n_experts = n_experts
        self.top_k = top_k
        self.balance_alpha = balance_alpha

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                n_experts=n_experts,
                top_k=top_k,
                balance_alpha=balance_alpha,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(
            d_model
        )

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = (
            self.token_embedding.weight
        )

        self.apply(
            self._init_weights
        )

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.Embedding):

            nn.init.xavier_uniform_(
                module.weight
            )

        elif isinstance(module, nn.RMSNorm):

            nn.init.ones_(
                module.weight
            )

    def forward(
        self,
        input_ids,
        return_aux_loss=False,
    ):

        x = self.token_embedding(
            input_ids
        )

        total_balance_loss = x.new_zeros(
            ()
        )

        for block in self.blocks:

            x, balance_loss = block(x)

            total_balance_loss = (
                total_balance_loss
                + balance_loss
            )

        x = self.final_norm(x)

        logits = self.lm_head(x)

        if return_aux_loss:

            return (
                logits,
                total_balance_loss,
            )

        return logits


In [7]:
class LoRMALinear(nn.Module):
    def __init__(self, original_linear, r=16, alpha=32, dropout=0.05, do_pi=True):
        super().__init__()
        self.original_linear = original_linear
        self.in_features = original_linear.in_features
        self.out_features = original_linear.out_features
        self.scaling, self.do_pi = alpha / r, do_pi
        self.dropout = nn.Dropout(dropout)
        for p in original_linear.parameters(): p.requires_grad = False
        W = original_linear.weight
        self.A = nn.Parameter(W.new_empty(r, self.out_features))
        self.B = nn.Parameter(W.new_zeros(self.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def effective_weight(self):
        W = self.original_linear.weight
        delta = self.B @ self.A
        if self.do_pi:
            rows = torch.arange(self.out_features, device=W.device)[:, None]
            cols = torch.arange(self.out_features, device=W.device)[None, :]
            delta = delta.gather(1, (cols - rows) % self.out_features)
        return W + self.scaling * (delta @ W)
    def forward(self, x):
        W = self.original_linear.weight
        W_eff = self.effective_weight()
        return F.linear(x, W, self.original_linear.bias) + F.linear(self.dropout(x), W_eff - W)


from torch.utils.checkpoint import checkpoint as activation_checkpoint
from contextlib import contextmanager

class CausalMoE(MoE):
    def forward(self, x, valid_mask=None):
        # Preserve the pretrained router and expert tensors, but route each token
        # independently so future target tokens and other batch rows cannot leak.
        B, T, D = x.shape
        flat = x.reshape(-1, D)
        valid = (torch.ones(B*T, dtype=torch.bool, device=x.device) if valid_mask is None
                 else valid_mask.reshape(-1).bool())
        probs = F.softmax(self.router(flat).float(), dim=-1)
        weights, indices = torch.topk(probs, self.top_k, dim=-1)
        weights = weights / weights.sum(dim=-1, keepdim=True).clamp_min(1e-9)
        output = torch.zeros_like(flat)
        # Dispatch only selected valid tokens to each expert.
        for expert_index, expert in enumerate(self.experts):
            token_ids, slots = ((indices == expert_index) & valid[:,None]).nonzero(as_tuple=True)
            if token_ids.numel() == 0: continue
            expert_output = expert(flat[token_ids])
            weighted = expert_output * weights[token_ids,slots,None].to(expert_output.dtype)
            output = output.index_add(0, token_ids, weighted.to(output.dtype))
        mean_probs = (probs * valid[:,None]).sum(dim=0) / valid.sum().clamp_min(1)
        balance_loss = self.balance_alpha * self.n_experts * mean_probs.square().sum()
        return output.view(B,T,D), balance_loss

class SummaryModel(EfficientDecoderLM):
    def forward(self, input_ids, valid_mask=None, return_aux_loss=False):
        x = self.token_embedding(input_ids)
        balance_sum = x.new_zeros(())
        for block in self.blocks:
            if self.training and USE_GRADIENT_CHECKPOINTING:
                x, balance = activation_checkpoint(block, x, valid_mask=valid_mask,
                    use_reentrant=False, preserve_rng_state=True)
            else: x, balance = block(x, valid_mask=valid_mask)
            balance_sum = balance_sum + balance
        logits = self.lm_head(self.final_norm(x))
        return (logits, balance_sum) if return_aux_loss else logits

    def forward_cached(self, input_ids, past=None):
        # Inference only. Keep RoPE positions relative to the complete prompt.
        x = self.token_embedding(input_ids)
        offset = 0 if past is None else past[0][0].size(2)
        T = input_ids.size(1)
        if offset + T > MAX_SEQ_LEN: raise ValueError('Generation exceeds Phase 1 context')
        new_past = []
        for i, block in enumerate(self.blocks):
            a = block.attention
            h = block.norm1(x)
            B = h.size(0)
            q = a.q_proj(h).view(B,T,a.n_heads,a.head_dim).transpose(1,2)
            k = a.k_proj(h).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
            v = a.v_proj(h).view(B,T,a.n_kv_heads,a.head_dim).transpose(1,2)
            cos = a.rope.cos[offset:offset+T][None,None]
            sin = a.rope.sin[offset:offset+T][None,None]
            def rotate(z):
                z1,z2 = z[...,::2],z[...,1::2]
                return torch.stack((z1*cos-z2*sin,z1*sin+z2*cos),dim=-1).flatten(-2)
            q,k = rotate(q),rotate(k)
            if past is not None:
                k = torch.cat((past[i][0],k),dim=2)
                v = torch.cat((past[i][1],v),dim=2)
            new_past.append((k,v))
            repeat = a.n_heads // a.n_kv_heads
            query_positions = (offset + torch.arange(T, device=x.device))[:,None]
            key_positions = torch.arange(k.size(2), device=x.device)[None,:]
            causal = key_positions <= query_positions
            local = key_positions >= query_positions - a.local_window + 1
            global_landmarks = (key_positions % a.global_stride) == 0
            mask = causal & (local | global_landmarks)
            out = F.scaled_dot_product_attention(q,k.repeat_interleave(repeat,dim=1),v.repeat_interleave(repeat,dim=1),attn_mask=mask[None,None])
            x = x + a.out_proj(out.transpose(1,2).contiguous().view(B,T,-1))
            moe_output, _ = block.moe(block.norm2(x))
            x = x + moe_output
        return self.lm_head(self.final_norm(x[:,-1:])), new_past

phase1_path = download_bucket_file(MODEL_PATH)

def build_summary_model():
    model = SummaryModel(VOCAB_SIZE, **MODEL_CONFIG)
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in checkpoint: state = checkpoint['model_state_dict']
    elif 'model' in checkpoint: state = checkpoint['model']
    elif 'state_dict' in checkpoint: state = checkpoint['state_dict']
    elif all(isinstance(v, torch.Tensor) for v in checkpoint.values()): state = checkpoint
    else: raise ValueError(f'Cannot identify weights: {list(checkpoint)}')
    # Restore the original vocabulary head too; no Phase 5 classification weights.
    state = dict(state)
    if 'lm_head.weight' not in state and 'token_embedding.weight' in state:
        state['lm_head.weight'] = state['token_embedding.weight']
    if not torch.equal(state['lm_head.weight'], state['token_embedding.weight']):
        raise ValueError('Checkpoint LM head and embedding disagree with the tied Phase 1 architecture')
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    for p in model.parameters(): p.requires_grad = False
    for block in model.blocks[-LORMA_LAST_N_BLOCKS:]:
        for name in ['q_proj', 'k_proj', 'v_proj', 'out_proj']:
            setattr(block.attention, name, LoRMALinear(getattr(block.attention, name), **ADAPTER_CONFIG))
    model = model.to(DEVICE)
    assert model.lm_head.weight is model.token_embedding.weight
    print('Total/trainable parameters:', sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))
    return model

def summary_loss_sum(model, batch):
    ids, labels = batch['input_ids'].to(DEVICE), batch['labels'].to(DEVICE)
    valid_mask = batch['valid_mask'].to(DEVICE)
    with autocast_context(): logits, balance = model(ids, valid_mask=valid_mask, return_aux_loss=True)
    labels = labels[:,1:].contiguous()
    count = int((labels != -100).sum())
    loss = F.cross_entropy(logits[:,:-1].float().reshape(-1,VOCAB_SIZE), labels.reshape(-1),
                           ignore_index=-100, reduction='sum')
    # Accumulation divides by target-token count, so scale the auxiliary mean accordingly.
    if model.training: loss = loss + balance.float() * count
    return loss, count

@torch.no_grad()
def generate_summary(model, prompt):
    model.eval()
    ids = torch.tensor([prompt],device=DEVICE,dtype=torch.long)
    past, generated = None, []
    for _ in range(MAX_NEW_TOKENS):
        with autocast_context(): logits, past = model.forward_cached(ids,past)
        next_id = int(logits[0,-1].argmax())
        if next_id == EOS_ID: break
        generated.append(next_id)
        ids = torch.tensor([[next_id]],device=DEVICE)
    return tokenizer.decode(generated, skip_special_tokens=True).strip(), len(generated)

@contextmanager
def merged_lorma_for_generation(model):
    # Materialize effective weights once, rather than on every generated token.
    # Restore the original modules (and optimizer parameter identities) afterwards.
    if model.training: raise RuntimeError('Merge is only for evaluation')
    originals = []
    try:
        with torch.no_grad():
            for block in model.blocks:
                for name in ['q_proj','k_proj','v_proj','out_proj']:
                    layer = getattr(block.attention,name)
                    if isinstance(layer,LoRMALinear):
                        weight = layer.effective_weight().detach()
                        merged = nn.Linear(layer.in_features,layer.out_features,bias=layer.original_linear.bias is not None,
                                           device=weight.device,dtype=weight.dtype)
                        merged.weight.copy_(weight)
                        if merged.bias is not None: merged.bias.copy_(layer.original_linear.bias)
                        merged.requires_grad_(False)
                        merged.eval()
                        originals.append((block.attention,name,layer))
                        setattr(block.attention,name,merged)
        yield
    finally:
        for attention,name,layer in originals: setattr(attention,name,layer)

@torch.no_grad()
def check_generation_cache(model):
    model.eval()
    # Verify the custom cache against the unchanged full-sequence decoder.
    ids = torch.tensor([datasets['train'].rows[0]['prompt'][:160]],device=DEVICE)
    with merged_lorma_for_generation(model):
        expected = model(ids)[:,-1:]
        prefill, past = model.forward_cached(ids[:,:-1])
        actual, _ = model.forward_cached(ids[:,-1:],past)
    torch.testing.assert_close(actual,expected,rtol=2e-4,atol=2e-4)
    print('Dual-attention cached/full-prefix logits agree on the startup check')

@torch.no_grad()
def check_moe_causality(model):
    model.eval()
    row = datasets['train'].rows[0]
    prefix = row['prompt'][:8]
    if not prefix: raise ValueError('Empty prompt')
    ids = torch.tensor([prefix], device=DEVICE)
    extended = torch.cat((ids, torch.tensor([[EOS_ID]], device=DEVICE)), dim=1)
    with merged_lorma_for_generation(model):
        original = model(ids)
        future_added = model(extended)[:,:ids.size(1)]
        torch.testing.assert_close(original, future_added, rtol=2e-4, atol=2e-4)
    print('MoE logits are independent of appended future tokens on the startup check')


(…)checkpoints/dual_attention_200M/model.pt:   0%|          | 0.00/712M [00:00<?, ?B/s]

In [8]:
def experiment_metadata(domain='SUMM'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                max_seq_len=MAX_SEQ_LEN, max_new_tokens=MAX_NEW_TOKENS, source_budget=SOURCE_BUDGET,
                source_spans=SOURCE_SPANS, source_selection='uniform_spans_with_gaps_v1', prompt_start=PROMPT_START, prompt_end=PROMPT_END,
                lorma_last_n_blocks=LORMA_LAST_N_BLOCKS, gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE), data_repo=DATA_REPO, data_prefix=DATA_PREFIX,
                data_fingerprint=DATA_FINGERPRINT, val_fraction=VAL_FRACTION, val_max_docs=VAL_MAX_DOCS,
                technique='LoRMA_identity_init', train_fraction=TRAIN_FRACTION,
                seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP, patience=PATIENCE, min_delta=MIN_DELTA,
                scheduler=dict(name='warmup_then_validation_plateau', warmup_ratio=WARMUP_RATIO,
                               start_factor=WARMUP_START_FACTOR, factor=LR_FACTOR, bad_checks=LR_BAD_CHECKS,
                               cooldown_checks=LR_COOLDOWN_CHECKS, min_lr=LR_MIN, loss_threshold=LR_LOSS_THRESHOLD,
                               checks_per_epoch=VAL_LOSS_CHECKS_PER_EPOCH, planned_epochs=NUM_EPOCHS),
                summary_key=SUMMARY_KEY, skip_empty_summaries=SKIP_EMPTY_SUMMARIES,
                attention='causal_local_global_landmarks', local_window=64, global_stride=128,
                moe_routing='per_token_topk', moe_auxiliary='valid_token_mean_probs_squared_train_only',
                selection_metric='rougeL_f1', decoding='greedy', loss='summary_only_token_mean')

def save_summary_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=model.state_dict(), optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_summary_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    model.load_state_dict(payload['model'])
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload


class ValidationLRScheduler:
    def __init__(self, optimizer, total_updates):
        assert total_updates > 0 and 0 < WARMUP_RATIO < 1
        assert 0 < WARMUP_START_FACTOR <= 1 and 0 < LR_FACTOR < 1
        assert LR_BAD_CHECKS >= 1 and 0 <= LR_MIN <= LR
        self.optimizer = optimizer
        self.updates = 0
        self.warmup_updates = max(1, math.ceil(total_updates * WARMUP_RATIO))
        self.peak_lrs = [g['lr'] for g in optimizer.param_groups]
        # PyTorch reduces after num_bad_epochs > patience: 1 means two bad checks.
        self.plateau = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=LR_FACTOR, patience=LR_BAD_CHECKS-1,
            threshold=LR_LOSS_THRESHOLD, threshold_mode='rel',
            cooldown=LR_COOLDOWN_CHECKS, min_lr=LR_MIN)
        self._set_warmup_lr()

    def _set_warmup_lr(self):
        factor = WARMUP_START_FACTOR + (1-WARMUP_START_FACTOR) * self.updates/self.warmup_updates
        for group, peak in zip(self.optimizer.param_groups, self.peak_lrs):
            group['lr'] = peak * factor

    def optimizer_step_completed(self):
        self.updates += 1
        if self.updates <= self.warmup_updates: self._set_warmup_lr()

    def validation_step(self, loss):
        if not math.isfinite(loss): raise ValueError('Non-finite validation loss')
        # Warmup and plateau never fight over the learning rate.
        if self.updates >= self.warmup_updates: self.plateau.step(loss)

    def state_dict(self):
        return dict(updates=self.updates, warmup_updates=self.warmup_updates,
                    peak_lrs=self.peak_lrs, plateau=self.plateau.state_dict(),
                    current_lrs=[g['lr'] for g in self.optimizer.param_groups])

    def load_state_dict(self, state):
        if state['warmup_updates'] != self.warmup_updates:
            raise ValueError('Warmup horizon changed; resume with the original dataset, epochs and accumulation')
        self.updates = state['updates']
        self.peak_lrs = state['peak_lrs']
        self.plateau.load_state_dict(state['plateau'])
        for group, lr in zip(self.optimizer.param_groups, state['current_lrs']): group['lr'] = lr

@torch.no_grad()
def evaluate_summary_loss(model, loader):
    was_training = model.training
    model.eval()
    try:
        total_loss = total_tokens = 0
        for batch in loader:
            loss, count = summary_loss_sum(model, batch)
            total_loss += loss.item(); total_tokens += count
        if total_tokens == 0: raise ValueError('No validation summary targets')
        return total_loss/total_tokens
    finally:
        model.train(was_training)

@torch.no_grad()
def evaluate_summary(model, loader):
    model.eval()
    validation_loss = evaluate_summary_loss(model, loader)
    scorer = rouge_scorer.RougeScorer(['rouge1','rougeL'],use_stemmer=True)
    records = []
    started = time.perf_counter()
    with merged_lorma_for_generation(model):
        for row in tqdm(datasets['dev'].rows,desc='Generate validation summaries'):
            prediction, tokens = generate_summary(model,row['prompt'])
            scores = scorer.score(row['reference'],prediction)
            words = prediction.lower().split()
            trigrams = list(zip(words,words[1:],words[2:]))
            records.append(dict(id=row['doc_id'],summary=prediction,reference=row['reference'],
                                rouge1_f1=scores['rouge1'].fmeasure,rougeL_f1=scores['rougeL'].fmeasure,
                                generated_tokens=tokens,repeated_trigram_fraction=1-len(set(trigrams))/len(trigrams) if trigrams else 0.0))
    metrics = {key:sum(r[key] for r in records)/len(records) for key in ['rouge1_f1','rougeL_f1','generated_tokens','repeated_trigram_fraction']}
    metrics.update(loss=validation_loss, generation_seconds=time.perf_counter()-started)
    with open(WORK_DIR / 'SUMM_dev_predictions.json','w',encoding='utf-8') as f:
        json.dump(records,f,ensure_ascii=False,indent=2)
    return metrics

def cleanup_latest_checkpoint():
    # Exact file deletion only. Never delete a parent directory or a Phase-1 file.
    best_local = WORK_DIR / 'checkpoints' / BEST_TAG / 'checkpoint.pt'
    if not best_local.is_file(): raise RuntimeError('Best checkpoint is missing; keeping latest')
    if UPLOAD_CHECKPOINTS:
        best_remote = checkpoint_remote_path(BEST_TAG)
        if not fs.exists(best_remote) or fs.info(best_remote)['size'] != best_local.stat().st_size:
            raise RuntimeError('Best upload not verified; keeping latest')
        latest_remote = checkpoint_remote_path(SAVE_TAG)
        assert latest_remote != best_remote and latest_remote != bucket_path(MODEL_PATH)
        if fs.exists(latest_remote): fs.rm(latest_remote)
        if fs.exists(latest_remote): raise RuntimeError('Latest bucket checkpoint deletion failed')
    latest_local = WORK_DIR / 'checkpoints' / SAVE_TAG / 'checkpoint.pt'
    if latest_local.exists(): latest_local.unlink()
    print('Training completed: retained best checkpoint only:', BEST_TAG)


In [9]:
def train_summary(domain='SUMM'):
    assert domain == 'SUMM'
    assert_run_destination()
    seed_everything(SEED)
    model = build_summary_model()
    check_generation_cache(model)
    check_moe_causality(model)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0 and VAL_LOSS_CHECKS_PER_EPOCH >= 1
    updates_per_epoch = math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
    check_interval = max(1, math.ceil(updates_per_epoch/VAL_LOSS_CHECKS_PER_EPOCH))
    scheduler = ValidationLRScheduler(optimizer, NUM_EPOCHS * updates_per_epoch)
    print('Optimizer updates per epoch:', updates_per_epoch, '| warmup updates:', scheduler.warmup_updates)
    start = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_summary_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        if 'scheduler' not in extra: raise ValueError('This checkpoint predates the scheduler; start a new run tag')
        scheduler.load_state_dict(extra['scheduler'])
        del checkpoint
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = seen_docs = 0
        next_log = LOG_EVERY_DOCS
        epoch_update = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            loss_sum, token_count = summary_loss_sum(model, batch)
            scaler.scale(loss_sum).backward()
            group_n += token_count
            train_n += token_count
            train_sum += loss_sum.item()
            seen_docs += len(batch['doc_ids'])
            if seen_docs >= next_log:
                print(f'Epoch {epoch+1} | docs {seen_docs:,}/{len(datasets["train"]):,} | training objective {train_sum/train_n:.5f}')
                next_log += LOG_EVERY_DOCS
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                previous_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                update_succeeded = scaler.get_scale() >= previous_scale
                if update_succeeded: scheduler.optimizer_step_completed()
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
                epoch_update += 1
                if epoch_update % check_interval == 0 or step + 1 == len(train_loader):
                    dev_loss = evaluate_summary_loss(model, dev_loader)
                    old_lr = optimizer.param_groups[0]['lr']
                    scheduler.validation_step(dev_loss)
                    current_lr = optimizer.param_groups[0]['lr']
                    event = dict(epoch=epoch+1, optimizer_updates=scheduler.updates,
                                 dev_loss=dev_loss, lr_before=old_lr, lr=current_lr)
                    extra.setdefault('lr_history', []).append(event)
                    print(f"Validation | update {scheduler.updates:,} | summary loss {dev_loss:.5f} | LR {old_lr:.3g} -> {current_lr:.3g}")
        metrics = evaluate_summary(model, dev_loader)
        f1 = metrics['rougeL_f1']
        improved = f1 > extra['best_f1']
        if improved:
            extra['best_f1'] = f1
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, lr=optimizer.param_groups[0]['lr'], **metrics))
        print(f"SUMM train loss={train_sum/train_n:.4f}, dev ROUGE-L F1={f1:.4f}, ROUGE-1 F1={metrics['rouge1_f1']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        extra['scheduler'] = scheduler.state_dict()
        if improved: save_summary_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_summary_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    best_payload = load_summary_checkpoint(best_tag, model, domain)
    del best_payload
    metrics = evaluate_summary(model, dev_loader)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    with open(WORK_DIR / 'SUMM_lr_history.json', 'w') as f: json.dump(extra.get('lr_history', []), f, indent=2)
    print('Best development metrics:', metrics)
    cleanup_latest_checkpoint()
    del optimizer
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id','summary'} and isinstance(p['summary'],str) for p in predictions)

@torch.no_grad()
def predict_summary(model):
    model.eval()
    predictions = []
    with merged_lorma_for_generation(model):
        for row in tqdm(datasets['test'].rows,desc='SUMM test predictions'):
            summary, _ = generate_summary(model,row['prompt'])
            predictions.append(dict(id=row['doc_id'],summary=summary))
    validate_predictions(predictions,data['test'])
    return predictions

summ_model, summ_history = train_summary()
# Load test only after training; test text is never used for fitting/selection.
data['test'] = normalize_documents(download_json(f'{DATA_PREFIX}/Test/test.json'),labelled=False)
datasets['test'] = SummaryDataset(data['test'],'test')
predictions = predict_summary(summ_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path,'w',encoding='utf-8') as f:
    json.dump(predictions,f,ensure_ascii=False,indent=2)
del summ_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 177942528 98304
Dual-attention cached/full-prefix logits agree on the startup check
MoE logits are independent of appended future tokens on the startup check
Optimizer updates per epoch: 835 | warmup updates: 209


SUMM epoch 1:   0%|          | 0/3338 [00:00<?, ?it/s]

Epoch 1 | docs 1,000/6,676 | training objective 6.61885
Validation | update 209 | summary loss 6.57883 | LR 0.0002 -> 0.0002
Epoch 1 | docs 2,000/6,676 | training objective 6.61619
Epoch 1 | docs 3,000/6,676 | training objective 6.60944
Validation | update 418 | summary loss 6.54566 | LR 0.0002 -> 0.0002
Epoch 1 | docs 4,000/6,676 | training objective 6.59989
Epoch 1 | docs 5,000/6,676 | training objective 6.59180
Validation | update 627 | summary loss 6.52736 | LR 0.0002 -> 0.0002
Epoch 1 | docs 6,000/6,676 | training objective 6.58452
Validation | update 835 | summary loss 6.51698 | LR 0.0002 -> 0.0002


Generate validation summaries:   0%|          | 0/100 [00:00<?, ?it/s]

SUMM train loss=6.5816, dev ROUGE-L F1=0.1490, ROUGE-1 F1=0.1601, patience=0/10


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (37 outputs left out)

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Tokenizing test:   0%|          | 0/100 [00:00<?, ?it/s]

test document tokens p50/p90/p99/max: [3734.5, 10241.1982421875, 33409.6875, 45355.0]
test documents needing source selection: 94


SUMM test predictions:   0%|          | 0/100 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/summ_runs/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau/prediction.json
ZIP: /kaggle/working/summ_runs/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau/summ_dual_attention_200M_lorma_len2048_summary512_spans8_tokenroute_warmup_plateau.zip
Cases: 100
